In [ ]:
# ──────────────────────────────────────────────────────────────
# Headers are in the code because they look nice
# "If my commit messages don't have emojis, how would you know how I feel?"
# - ProgrammersAreAlsoHuman, '0.1x engineers'
# ──────────────────────────────────────────────────────────────

"""This code trains several models to perform a 3SUM task generated
by https://github.com/JacobPfau/fillerTokens/tree/master, comparing
model training curves, attention maps, and test set performance in
order to highlight the role of filler tokens."""


In [1]:
# ──────────────────────────────────────────────────────────────
# Imports
# ──────────────────────────────────────────────────────────────
import datetime as dt
import itertools
import json
import os
import pathlib
import re
import shutil
import textwrap
from pathlib import Path
from glob import glob
from typing import Final
from types import SimpleNamespace as ns
from collections import Counter

import numpy as np
import pandas as pd
import polars as pl
import torch
from datasets import load_dataset
from google.colab import drive
from huggingface_hub import notebook_login
from peft import LoraConfig, get_peft_model
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)


In [3]:
# ──────────────────────────────────────────────────────────────
# Global variables and namespace for the run
# ──────────────────────────────────────────────────────────────
# Output paths
BASE_DIR = "/content/drive/MyDrive/Colab_Files/repurposed_tokens"
RUN_DIR = os.path.join(BASE_DIR, RUN)
os.makedirs(RUN_DIR, exist_ok=True)

# Run namespace
presets = {
    'direct': ns(name='direct',
                 filler_tok=''),
    'cot': ns(name='cot',
              filler_tok=range(0, 1000)),
    'dot': ns(name='dot',
              filler_tok='.'),
    'low_S': ns(name='low_S',
                filler_tok='big'),
    'high_S': ns(name='high_S',
                 filler_tok='neu')
}
RUN = presets['direct']

# Model
MODEL_ID = "meta-llama/Meta-Llama-3-8B-instruct"
MODEL_DIR = os.path.join(RUN_DIR, RUN.name)
os.makedirs(MODEL_DIR, exist_ok=True)


SyntaxError: invalid syntax. Perhaps you forgot a comma? (<ipython-input-3-4041716477>, line 9)

In [ ]:
# ──────────────────────────────────────────────────────────────
# Github and Drive setup
# ──────────────────────────────────────────────────────────────
%cd /content

# Mount drive for data
drive.mount('/content/drive', force_remount=True)

# Filepaths
FILLER_DIR = "fillerTokens"
ENTROPIX_DIR = "entropix"

# Clone and setup repos
if not os.path.exists(FILLER_DIR):
    !git clone https://github.com/BreckEmert/fillerTokens.git
    !pip install -r {FILLER_DIR}/requirements.txt

if not os.path.exists(ENTROPIX_DIR):
    !git clone https://github.com/BreckEmert/entropix.git

# Create symlink to data
if not os.path.islink('./data'):
    !ln -s {RUN_DIR} ./data

# Set HF cache
os.environ['HF_HOME'] = 'drive/MyDrive/HF_cache'


In [ ]:
# ──────────────────────────────────────────────────────────────
# HF setup
# ──────────────────────────────────────────────────────────────
notebook_login()


# Data Generation

In [ ]:
"""
I plan to use these training runs:
    1) CoT as filler (filler unmasked)
    2) Dots as filler (filler masked out)
    3) Low entropy tokens as filler (filler masked out)
    4) High entropy tokens as filler (filler masked out)
    5) Direct-to-answer

This allows for
    1) A high-quality gold standard for the model.  Not very related
        to my experiment but a super-baseline.
    2) Progressively more difficult tasks to highlight the effect of
        filler tokens without the large amount of runs that are
        theoretically all relevant here, as we step through runs 2-4.


Outline of data pipeline for clarity:
1) We generate three kinds of samples, direct, CoT and punct:
    533 569 530 814 A False     VERIFY THIS DOESNT HAVE P AND A IM NOT SURE!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
    371 578 006 519 P 1- 4 0- 7 3- 7- 4- 4 5- 6 A True
    873 545 827 245 P . . . . . . . . . . . . . A False
    # Note that this is prompt followed by answer in the same string
    # Answers are one word long (True/False), so
      future [-1], len(prompt)-1, are referring to this
2) Turn that into a HF dataset which has two columns:
    a) input_ids: tokenized full prompts
    b) labels: same except -100-masked everywhere but answer and eos tokens
3) Lastly, the collator recieves a list of those dicts and
    a) pads batches to the longest of that batch
    b) stacks examples into tensors
    c) leaves labels alone
"""

# ──────────────────────────────────────────────────────────────
# Generate raw math data with scripts.data_match3
# ──────────────────────────────────────────────────────────────
%cd {FILLER_DIR}  # must be run because the script is a module

# Generate the datasets if they don't exist
settings = {
    "direct": dict(cot_rate=0, no_filler_rate=1),
    "cot": dict(cot_rate=1, no_filler_rate=0),
    "dot": dict(cot_rate=0, no_filler_rate=0)
}  # (punct is 100% of generation if both are 0 by default)

# Loop through each dataset type
for name, rates in settings.items():
    # Get output paths and see if they exist
    output_dir = os.path.join(BASE_DIR, name)
    os.makedirs(output_dir, exist_ok=True)

    train_path = f"{output_dir}/demo3s_trainset.csv"
    test_path = f"{output_dir}/demo3s_testset.csv"
    cfg_path = f"{output_dir}/demo3s_config.json"

    if bool(glob(train_path)) and bool(glob(test_path)):
        continue

    # Run the data generation script
    !python -m scripts.data_match3 \
        --name demo3s \
        --length 12 \
        --dimension 3 \
        --train_samples 500000 \
        --test_samples 10000 \
        --cot_rate {rates['cot_rate']} \
        --no_filler_rate {rates['no_filler_rate']} \
        --corruption_rate 1.33
    # Args:
        # --length 12  # number of tuples per instance
        # --dimension 3  # number of integers per tuple
        # --cot_rate 0.5  # fraction to annotate with CoT traces
        # --no_filler_rate 0  # 0 so every non-CoT is punctuation-filler
        #  --corruption_rate 1.33  # False instances have avg 1.33 digits randomly replaced (False are generated from True examples and made False by replacing digits)
    # Generates:
        # args_demo3s_YYYY-MM-DD.json  # metadata
        # demo3s_trainset_YYYY-MM-DD.csv
        # demo3s_testset_YYYY-MM-DD.csv

    # Remove the date from their names
    raw_train_csv = glob(f"{output_dir}/demo3s_trainset_*.csv")[0]
    raw_test_csv = glob(f"{output_dir}/demo3s_testset_*.csv")[0]
    raw_cfg_csv = glob(f"{output_dir}/demo3s_testset_*.csv")[0]

    Path(raw_train_csv).rename(train_path)
    Path(raw_test_csv).rename(test_path)
    Path(raw_cfg_csv).rename(cfg_path)

    # Saving them permanently into Drive
    for file in glob("data/demo3s_*"):
        shutil.copy(file, RUN_DIR)

# Find the newest files made, grabbing by mtime for safety
raw_train_csv = os.path.join(RUN_DIR, "demo3s_trainset.csv")
raw_test_csv  = os.path.join(RUN_DIR, "demo3s_testset.csv")
cfg_json = os.path.join(f"{output_dir}/demo3s_config.json")

print(f"Train csv: {raw_train_csv}\n")
print(f"Test csv: {raw_test_csv}\n")
print(f"Config json: {open(cfg_json).read()}")

!ls {RUN_DIR}


In [ ]:
# ──────────────────────────────────────────────────────────────
# Generate low-S and high-S datasets as well
# ──────────────────────────────────────────────────────────────
def replace_filler(prompt: str, filler_token: str) -> str:
    # Get segments of the example
    question, right = prompt.split(" P", 1)
    filler, answer = prompt.split(" A", 1)

    # Make the new filler
    n_tokens = len(filler.split())
    new_filler = " ".join(filler_token for _ in range(n_tokens))

    # Return the original q/a with the new filler
    return f"{question} P {new_filler} A{answer}"  # ALSO NEED TO VERIFY NO SPACE AFTER 'A'

if FILLER and FILLER in ["low_S", "high_S"]:
    for split in [raw_train_csv, raw_test_csv]:
        output_path = f"{split[:-4]}_{FILLER}_{FILLER_TOK}.csv"
        df = (
            pl.scan_csv(split)
            .with_columns(
                pl.col('prompt')
                .map_elements(lambda s: replace_filler(s, filler_token))
            )
        )
        df.write_csv(output_path)

        print(f"{FILLER} dataset generated:")
        print(df.head())


In [ ]:
# ──────────────────────────────────────────────────────────────
# Prepare raw math data into a HF dataset
# ──────────────────────────────────────────────────────────────
# Grab the data
def load_csv(path, split):
    return load_dataset(
        'csv',
        data_files=path,
        split=split,
        header=None,
        column_names=['prompt'],
    )
train_raw = load_csv(raw_train_csv, 'train')
test_raw = load_csv(raw_test_csv, 'test')

# Tokenise and make a loss mask
    # We only want loss on the answer token and eos, because we
    # want it to use the periods how it wants, not generate them.
tok = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
tok.pad_token = tok.eos_token

RUN.filler_ids = torch.tensor(tok(f" {RUN.filler_tok}")['input_ids'])
COT_BEGIN_TOK = tok(" P")['input_ids']
TRUE_TOK = tok(" True")['input_ids']
FALSE_TOK = tok(" False")['input_ids']
assert len(COT_BEGIN_TOK + TRUE_TOK + FALSE_TOK) == 3

def tokenize_and_mask(batch):
    # Tokenize the batch
    ids_batch = tok(batch['prompt'], add_special_tokens=True)['input_ids']

    # Mask off everything but answer and EOS
    labels = []
    for ids in ids_batch:
        label = [-100]*len(ids)
        label[-2:] = ids[-2:]
        labels.append(label)

    return {'input_ids': ids_batch, 'labels': labels}

    # Note to me learning HF: batch has nothing to do with model batches, this is just cpu efficiency
train_ds = train_raw.map(tokenize_and_mask, batched=True, remove_columns=['prompt'])
test_ds = test_raw.map(tokenize_and_mask, batched=True, remove_columns=['prompt'])  # COMPLETELY UNSURE ON THIS REMOVE_COL NAME CHECK IT

# Collation is done in the finetune step, which just pads


Train variant counts:  variant
cot      250061
punct    249939
Name: count, dtype: int64
Test variant counts:  variant
punct    5034
cot      4966
Name: count, dtype: int64
Saving punct subset
Saving cot subset

We now have five files:
train_punct.csv   test_punct.csv
train_cot.csv     test_cot.csv
train_mini.csv



# Finetune

In [ ]:
"""
I plan to use these training runs:
    1) CoT as filler (filler unmasked)
    2) Dots as filler (filler masked out)
    3) Low entropy tokens as filler (filler masked out)
    4) High entropy tokens as filler (filler masked out)
    5) Direct-to-answer

This allows for
    1) A high-quality gold standard for the model.  Not very
        related to my experiment but a super-baseline.
    2) Progressively more difficult tasks to highlight the
        effect of filler tokens without the large amount of
        runs that are theoretically all relevant here, as
        we step through runs 2-4.
"""
# ──────────────────────────────────────────────────────────────
# 3. Attach LoRA adapter to the model
# ──────────────────────────────────────────────────────────────
# Load model and resize embeddings
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype='auto',
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    device_map='auto',
)
base_model.resize_token_embeddings(len(tok))

# LoRA config
lora_cfg = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
)
model = get_peft_model(base_model, lora_cfg)
model.print_trainable_parameters()


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

trainable params: 6,815,744 || all params: 8,037,076,992 || trainable%: 0.0848


In [ ]:
# ──────────────────────────────────────────────────────────────
# 4. Finetune
# ──────────────────────────────────────────────────────────────
args = TrainingArguments(
    output_dir='ckpt_punct',
    label_names=['labels'],
    num_train_epochs=1,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,  # 8*4 = 32 examples per optimizer step
    learning_rate=2e-4,
    fp16=True,
    optim='paged_adamw_8bit',
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='no',
    report_to='none',  # DISABLE WANDB FOR QUICK RUNS
)

# Just pad/stacks
data_collator = DataCollatorWithPadding(
    tokenizer=tok,
    pad_to_multiple_of=8,  # Only useful on A100
    return_tensors='pt',  # I'm from tf so I'll be typing this :D
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    data_collator=data_collator,
)

# Train
trainer.train()
model.save_pretrained(MODEL_DIR)
tok.save_pretrained(MODEL_DIR)


No label_names provided for model class `PeftModel`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


OutOfMemoryError: CUDA out of memory. Tried to allocate 682.00 MiB. GPU 0 has a total capacity of 39.56 GiB of which 14.88 MiB is free. Process 28560 has 39.53 GiB memory in use. Of the allocated memory 38.57 GiB is allocated by PyTorch, and 475.45 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

# Evaluation and Visualization

In [ ]:
# ──────────────────────────────────────────────────────────────
# Load the finetuned model and fuse in the LoRA adapter
# ──────────────────────────────────────────────────────────────
# Base model
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,                 # e.g. "meta-llama/Meta-Llama-3-8B-instruct"
    torch_dtype="auto",
    device_map="auto",
)

# Finetuned LoRA
model = PeftModel.from_pretrained(
    base_model,
    MODEL_DIR,
)

# Merge
model = model.merge_and_unload()
model.eval()


In [ ]:
# ──────────────────────────────────────────────────────────────
# Attention hook
# ──────────────────────────────────────────────────────────────
# Globals that get filled in the pass
ATTN_BUCKET, CURRENT_FILL = [], None

def record_attn(module, input, output):
    """Writes (B, H) into CURRENT_FILL (B, L, H)"""
    attention_probs = output[1]  # (B, H, Q, K)

    filler_mask = torch.isin(TOKEN_IDS_THIS_BATCH, RUN.filler_ids)  # (B, S)
    filler_mask = filler_mask.float().unsqueeze(1).unsqueeze(2)  # (B, 1, 1, S)
        # B matches
        # 1 broadcasts across all heads
        # 1 broadcasts across all query positions
        # S matches the source/key length K

    # Compute mean over query dimension
    fill = (attention_probs * filler_mask).sum(-1).mean(-1)  # (B, H)
    CURRENT_FILL[:, module.layer_index, :] = fill.cpu()

# Attach to every layer
def attach_attention_hooks(model):
    hook_handles = []
    for i, block in enumerate(model.model.layers):
        block.self_attn.layer_index = i

        handle = block.self_attn.register_forward_hook(
            lambda module, input, output, idx=i:  # Using i makes the lambda distinct otherwise all hooks would just be the same, because they'd take from the last lambda in the for loop.
            record_attn(module, input, output)
        )
        hook_handles.append(handle)
    return hook_handles

hook_handles = attach_attention_hooks(model)


In [ ]:
# ──────────────────────────────────────────────────────────────
# Attention logging and visualization
# ──────────────────────────────────────────────────────────────
def get_raw_attention(dataloader):
    ATTN_BUCKET.clear()
    L = model.config.num_hidden_layers
    H = model.config.num_attention_heads

    with torch.no_grad():
        for batch in dataloader:
            global TOKEN_IDS_THIS_BATCH, CURRENT_FILL
            TOKEN_IDS_THIS_BATCH = batch["input_ids"]  # (B, S)

            B = TOKEN_IDS_THIS_BATCH.size(0)
            CURRENT_FILL = torch.zeros(B, L, H)  # (B, L, H)  # IF THIS THROWS A TYPE ERROR SET dtype=attention_probs.dtype not sure
            _ = model(
                **{k: v.to("cuda") for k, v in batch.items()},
                use_cache=False
            )
            ATTN_BUCKET.append(CURRENT_FILL)

    raw_attn = torch.cat(ATTN_BUCKET, dim=0)  # (N, L, H)
    return raw_attn

def calculate_enrichment(raw_attn):
    """Adjusts the attention for the percentage of filler tokens
    in the ds otherwise the attention mass is biased
    """
    # Calculate the filler rate
    n_fill, n_total = 0, 0
    for ex in dataset:
        ids = ex["input_ids"]
        n_total += len(ids)
        n_fill += torch.isin(torch.tensor(ids), RUN.filler_ids).sum().item()

    fill_rate = n_fill / n_total if n_total else 0.0
    print(f"Corpus filler rate: {fill_rate:.3%}")

    # Calculate the enrichment
    prob_to_fill = raw_attn.mean(dim=0)  # (L, H)
    return prob_to_fill / max(fill_rate, 1e-9)

def plot_enrichment(mat, title=''):
    """Quick imshow heat-map."""
    import matplotlib.pyplot as plt
    plt.imshow(mat.numpy(), aspect='auto')
    plt.xlabel("Head")
    plt.ylabel("Layer  (0 = bottom)")
    plt.title(title)
    plt.colorbar(label="× corpus filler rate")
    plt.show()

raw_attn = get_raw_attention(test_ds)  # (N, L, H)
enr = enrichment_matrix(raw_attn, FILL_RATE)
plot_enrichment(enr, f"Enrichment – run: {RUN.name}")
np.save(os.path.join(RUN_DIR, "enrichment.npy"), enr.numpy())


In [ ]:
# ──────────────────────────────────────────────────────────────
# Accuracy Eval
# ──────────────────────────────────────────────────────────────
# Remove these if they're still on
for handle in hook_handles:
    handle.remove()

def accuracy(model) -> float:
    """Evaluates model accuracy on test set"""
    # This just needs drastically improved.  I don't pass in the test set yet, and it's also very basic.  I should probably be doing something other than raw accuracy I just don't know what?
    # Iterate over test set
    invalid, correct = Counter(), 0
    with torch.no_grad():
        for ex in tqdm(test_ds, leave=False):
            prompt = tok.decode(ex['input_ids'], skip_special_tokens=True)

            # Forward pass
            enc = {k: v.to('cuda')
                   for k, v in tok(prompt, return_tensors='pt').items()}
            out = model.generate(**enc, max_new_tokens=1, do_sample=False)

            # Check answer
            pred = tok.decode([out[0, -1].item()]).strip()
            exp = prompt.rstrip().split()[-1]  # validate this based on input
            if pred == exp:
                correct += 1
            else:
                invalid[pred] += 1

    accuracy = correct / len(test_ds)
    return accuracy, invalid

# this is off now that I don't have dots/no dots since that would be handled by aggregating several runs of data, not something we do all at once necessarily?  I guess we can do it all at once after all the models are trained, but it seems useful to have some sort of utility function that works as we go so we can tune the runs.
acc_dots, invalid_dots = accuracy(model)
acc_nodots, invalid_nodots = accuracy(model)

print(f"Ambiguous with dots: {sum(invalid_dots)}")
print(f"Ambiguous no dots  : {sum(invalid_nodots)}")
print(f"Accuracy with dots : {acc_dots:.3f}")
print(f"Accuracy no dots   : {acc_nodots:.3f}")
print(f"delta              : {acc_dots - acc_nodots:+.3f}")
